# PM2.5 Data Preprocessing Pipeline — Los Angeles County

**Project:** DSA SCI 8001 Time Series Analysis — Final Project  
**Dataset:** PurpleAir sensor data (California), filtered to Los Angeles County  
**Time Span:** 2017–2023  
**Output:** `la_daily_2017_2023.csv`

---

## Pipeline Overview

Raw PurpleAir data is provided at the California state level. The data format differs by year:
- **2017–2020**: One CSV file per year (`pa_ca_YYYY.csv`)
- **2021–2023**: One folder per year containing monthly CSV files (`pa_ca_YYYY/`)

For 2021–2023, we **geo-filter each monthly file first**, then merge — this is much faster than merging all of California first (e.g. pa_ca_2021 is ~10 GB).

| Step | Description | Applies To |
|------|-------------|------------|
| 1 | **Geo-filter** annual CSV → LA County | 2017–2020 |
| 2 | **Geo-filter each monthly file** → LA, then merge into annual | 2021–2023 |
| 3 | **QAQC + Daily aggregation** across all sensors | All years (2017–2023) |

---

## Directory Structure Expected

```
pm2.5/
├── pm/
│   ├── pa_ca_2017.csv         # Single annual file
│   ├── pa_ca_2018.csv
│   ├── pa_ca_2019.csv
│   ├── pa_ca_2020.csv
│   ├── pa_ca_2021/            # Folder of monthly files
│   ├── pa_ca_2022/            # Folder of monthly files
│   └── pa_ca_2023/            # Folder of monthly files
├── pm_LA_final/               # Created by this script
├── geo_dataset/
│   └── ca_county.shp
└── final_project/
    └── data/                  # Final output saved here
```

## 0. Setup & Imports

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd

# ── Configure your base path here ──────────────────────────────────────────
BASE_PATH       = "/Users/wanxinli/Desktop/pm2.5/"

RAW_FOLDER      = os.path.join(BASE_PATH, "pm")
LA_FINAL_FOLDER = os.path.join(BASE_PATH, "pm_LA_final")
OUTPUT_FOLDER   = os.path.join(BASE_PATH, "final_project", "data")
LA_SHP_PATH     = os.path.join(BASE_PATH, "geo_dataset", "ca_county.shp")

os.makedirs(LA_FINAL_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER,   exist_ok=True)

YEARS_SINGLE  = [2017, 2018, 2019, 2020]   # single annual CSV
YEARS_MONTHLY = [2021, 2022, 2023]          # folder of monthly CSVs
ALL_YEARS     = YEARS_SINGLE + YEARS_MONTHLY

print("Setup complete. Output folder:", OUTPUT_FOLDER)

Setup complete. Output folder: /Users/wanxinli/Desktop/pm2.5/final_project/data


---
## Shared Helper: Geo-filter Function

Used by both Step 1 and Step 2. Reads a California-wide CSV (in chunks to handle large files) and retains only rows within Los Angeles County via spatial join.

In [2]:
def filter_la_county(input_file, output_file, la_boundary, chunksize=200_000):
    """
    Spatially filter a California-wide PurpleAir CSV to retain only
    rows whose (latitude, longitude) fall within Los Angeles County.

    Parameters
    ----------
    input_file  : str           – Path to the input California CSV
    output_file : str           – Destination path for the LA-only output CSV
    la_boundary : GeoDataFrame  – LA County boundary polygon (pre-loaded, EPSG:4326)
    chunksize   : int           – Rows per iteration to avoid memory overflow
    """
    first_chunk = True

    for chunk in pd.read_csv(input_file, chunksize=chunksize, low_memory=False):
        chunk = chunk.dropna(subset=["latitude", "longitude"])

        gdf = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk.longitude, chunk.latitude),
            crs="EPSG:4326"
        )

        filtered = gpd.sjoin(gdf, la_boundary[["geometry"]], predicate="within")
        filtered = filtered.drop(columns=["geometry", "index_right"])

        filtered.to_csv(output_file,
                        mode="w" if first_chunk else "a",
                        header=first_chunk,
                        index=False)
        first_chunk = False


# Load LA County boundary once — reused across all steps
ca_counties = gpd.read_file(LA_SHP_PATH)
la_boundary = ca_counties[ca_counties["NAME"] == "Los Angeles"].to_crs("EPSG:4326")
print("LA County boundary loaded.")

LA County boundary loaded.


---
## Step 1 — Geo-filter Annual Files (2017–2020)

Each year is a single CSV covering all of California. We filter directly to LA County using chunked reading to handle file sizes up to ~4 GB.

In [3]:
for y in YEARS_SINGLE:
    input_file  = os.path.join(RAW_FOLDER,      f"pa_ca_{y}.csv")
    output_file = os.path.join(LA_FINAL_FOLDER, f"pa_la_{y}.csv")

    print(f"{y}: geo-filtering {os.path.basename(input_file)}...")
    filter_la_county(input_file, output_file, la_boundary)
    print(f"  → Saved: {output_file}")

print("\nStep 1 complete.")

2017: geo-filtering pa_ca_2017.csv...
  → Saved: /Users/wanxinli/Desktop/pm2.5/pm_LA_final/pa_la_2017.csv
2018: geo-filtering pa_ca_2018.csv...
  → Saved: /Users/wanxinli/Desktop/pm2.5/pm_LA_final/pa_la_2018.csv
2019: geo-filtering pa_ca_2019.csv...
  → Saved: /Users/wanxinli/Desktop/pm2.5/pm_LA_final/pa_la_2019.csv
2020: geo-filtering pa_ca_2020.csv...
  → Saved: /Users/wanxinli/Desktop/pm2.5/pm_LA_final/pa_la_2020.csv

Step 1 complete.


---
## Step 2 — Geo-filter Monthly Files, Then Merge (2021–2023)

For 2021–2023, data comes as monthly files inside a yearly folder (e.g. `pa_ca_2021/`). Each monthly file is individually filtered to LA County **before** merging.

**Why filter first, then merge?**  
Each monthly California file is ~0.5–1 GB. Filtering first reduces each file to just LA County sensors (much smaller), so the final merge is fast. The alternative — merging all months first (~10 GB) then filtering — is much slower and memory-intensive.

In [4]:
for y in YEARS_MONTHLY:
    folder_path   = os.path.join(RAW_FOLDER, f"pa_ca_{y}")
    monthly_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".csv")])

    print(f"{y}: found {len(monthly_files)} monthly files — filtering each to LA County...")

    la_monthly_dfs = []

    for f in monthly_files:
        input_file  = os.path.join(folder_path, f)
        tmp_output  = os.path.join(LA_FINAL_FOLDER, f"tmp_la_{f}")

        # Geo-filter this month to LA County
        filter_la_county(input_file, tmp_output, la_boundary)

        # Read the small LA-only file back into memory
        df_month = pd.read_csv(tmp_output, low_memory=False)
        la_monthly_dfs.append(df_month)

        # Remove temporary file
        os.remove(tmp_output)
        print(f"  {f}: {len(df_month):,} LA rows")

    # Merge all LA-filtered months into one annual file
    df_annual = pd.concat(la_monthly_dfs, ignore_index=True)
    df_annual = df_annual.sort_values("date_hour")

    output_file = os.path.join(LA_FINAL_FOLDER, f"pa_la_{y}.csv")
    df_annual.to_csv(output_file, index=False)
    print(f"  → Annual file saved: {output_file}  ({len(df_annual):,} total rows)\n")

print("Step 2 complete.")

2021: found 12 monthly files — filtering each to LA County...
  pa_ca_2021_01.csv: 391,531 LA rows
  pa_ca_2021_02.csv: 370,789 LA rows
  pa_ca_2021_03.csv: 421,435 LA rows
  pa_ca_2021_04.csv: 421,744 LA rows
  pa_ca_2021_05.csv: 446,704 LA rows
  pa_ca_2021_06.csv: 434,373 LA rows
  pa_ca_2021_07.csv: 452,168 LA rows
  pa_ca_2021_08.csv: 465,653 LA rows
  pa_ca_2021_09.csv: 468,979 LA rows
  pa_ca_2021_10.csv: 492,249 LA rows
  pa_ca_2021_11.csv: 482,272 LA rows
  pa_ca_2021_12.csv: 502,157 LA rows
  → Annual file saved: /Users/wanxinli/Desktop/pm2.5/pm_LA_final/pa_la_2021.csv  (5,350,054 total rows)

2022: found 12 monthly files — filtering each to LA County...
  pa_ca_2022_01.csv: 513,640 LA rows
  pa_ca_2022_02.csv: 475,226 LA rows
  pa_ca_2022_03.csv: 532,776 LA rows
  pa_ca_2022_04.csv: 523,364 LA rows
  pa_ca_2022_05.csv: 547,322 LA rows
  pa_ca_2022_06.csv: 523,935 LA rows
  pa_ca_2022_07.csv: 530,456 LA rows
  pa_ca_2022_08.csv: 522,786 LA rows
  pa_ca_2022_09.csv: 507,699 LA

---
## Step 3 — QAQC & Daily Aggregation

Loads all seven LA-filtered annual files, applies quality control, and aggregates to a **daily, county-wide** time series.

### Quality Control (QAQC)
| Rule | Threshold | Rationale |
|------|-----------|-----------|
| Remove extreme PM2.5 | `> 500 µg/m³` | Physically implausible for ambient air; likely sensor malfunction |

### Output Columns
| Column | Description |
|--------|-------------|
| `pm25_cf_1` | Daily mean PM2.5 (CF=1 correction), µg/m³ |
| `humidity` | Daily mean relative humidity (%) |
| `temperature` | Daily mean temperature (°F) |
| `pressure` | Daily mean atmospheric pressure |

In [5]:
df_list = []

for y in ALL_YEARS:
    path = os.path.join(LA_FINAL_FOLDER, f"pa_la_{y}.csv")
    df   = pd.read_csv(path, low_memory=False)
    df["date_hour"] = pd.to_datetime(df["date_hour"])

    # QAQC: flag physically implausible PM2.5 readings as NaN
    n_flagged = (df["pm25_cf_1"] > 500).sum()
    df.loc[df["pm25_cf_1"] > 500, "pm25_cf_1"] = np.nan
    print(f"{y}: {len(df):,} rows | {n_flagged:,} readings flagged (>500 µg/m³)")

    df_list.append(df)

full_df = pd.concat(df_list, ignore_index=True)
full_df = full_df.sort_values("date_hour")

print(f"\nTotal rows: {len(full_df):,}")
print(f"Date range: {full_df['date_hour'].min()} → {full_df['date_hour'].max()}")

2017: 77,259 rows | 375 readings flagged (>500 µg/m³)
2018: 642,050 rows | 580 readings flagged (>500 µg/m³)
2019: 1,359,943 rows | 5,238 readings flagged (>500 µg/m³)
2020: 2,622,782 rows | 19,459 readings flagged (>500 µg/m³)
2021: 5,350,054 rows | 48,606 readings flagged (>500 µg/m³)
2022: 6,237,198 rows | 67,086 readings flagged (>500 µg/m³)
2023: 2,059,818 rows | 20,038 readings flagged (>500 µg/m³)

Total rows: 18,349,104
Date range: 2017-01-01 00:00:00 → 2023-10-17 03:00:00


In [6]:
# ── Daily aggregation: average all LA sensors per calendar day ──────────────
full_df["date"] = full_df["date_hour"].dt.date

agg_dict = {"pm25_cf_1": "mean"}
for var in ["humidity", "temperature", "pressure"]:
    if var in full_df.columns:
        agg_dict[var] = "mean"

daily = (
    full_df
    .groupby("date")
    .agg(agg_dict)
    .reset_index()
)

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").set_index("date")

print("Missing values after aggregation:")
print(daily.isnull().sum())
print(f"\nTotal days: {len(daily)}")
print(daily.head())

Missing values after aggregation:
pm25_cf_1        0
humidity         0
temperature      0
pressure       342
dtype: int64

Total days: 2481
            pm25_cf_1   humidity  temperature   pressure
date                                                    
2017-01-01  25.714433  54.929449    59.433115  59.433218
2017-01-02   7.568054  53.618319    61.723596  61.726110
2017-01-03  18.169412  52.460524    62.956328  62.954285
2017-01-04  21.875172  51.506422    63.944057  63.943135
2017-01-05  16.650620  65.335559    65.901941  65.899936


In [7]:
# ── Save final output ───────────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_FOLDER, "la_daily_2017_2023.csv")
daily.to_csv(output_path)
print(f"Saved: {output_path}")
print(f"Shape: {daily.shape}")
print("\nPreprocessing pipeline complete!")

Saved: /Users/wanxinli/Desktop/pm2.5/final_project/data/la_daily_2017_2023.csv
Shape: (2481, 4)

Preprocessing pipeline complete!


---
## Summary

| Step | Operation | Input | Output |
|------|-----------|-------|--------|
| 1 | Geo-filter CA → LA | `pm/pa_ca_YYYY.csv` (2017–2020) | `pm_LA_final/pa_la_YYYY.csv` |
| 2 | Geo-filter each month → LA, then merge | `pm/pa_ca_YYYY/*.csv` (2021–2023) | `pm_LA_final/pa_la_YYYY.csv` |
| 3 | QAQC + daily aggregation | `pa_la_YYYY.csv` (all years) | `la_daily_2017_2023.csv` |

The final output `la_daily_2017_2023.csv` contains a clean daily time series for Los Angeles County from **2017 to 2023**, with PM2.5 as the target variable and humidity, temperature, and pressure as exogenous regressors.